## Script to parallelize the forward pass across the population huge speed improvements

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import sys
sys.path.append('C:/Users/Admin/Desktop/PhD/simulation/simulation_python/nonlinear_oscillator_network/Utils')

from NN_utils import *
import torch
import torch.nn as nn
from torchvision import datasets, transforms
#from torchsummary import summary
import time
from types import SimpleNamespace
import pickle
import gc
from optimization_algorithms import *

from functorch import make_functional_with_buffers
from functorch import vmap

In [2]:
transform_data = transforms.Compose([
    transforms.ToTensor()
    #transforms.Normalize((0.2868,), (0.3524,))
])

MNIST_train = datasets.MNIST(root='./data', train=True, transform=transform_data, download=True)
MNIST_test = datasets.MNIST(root='./data', train=False, transform=transform_data, download=True)

train_loader_MNIST = torch.utils.data.DataLoader(dataset=MNIST_train, batch_size=1000, shuffle=True)
test_loader_MNIST = torch.utils.data.DataLoader(dataset=MNIST_test, batch_size=10000, shuffle=False)

X_train_MNIST, Y_train_MNIST = next(iter(train_loader_MNIST))
X_test_MNIST, Y_test_MNIST = next(iter(test_loader_MNIST))

In [3]:
n_neurons = 30

RNN_params = {
        "N_in": 784,               # e.g., flattened 28x28 FashionMNIST image
        "N_out": 10,               # number of classes in FashionMNIST
        "N_neurons": n_neurons,          # number of hidden units per RNN layer
        "N_layers": 3,             # depth of the RNN
        "time_steady_state": 200    # number of repeated timesteps to reach steady state
    }
    
model = Oscillator_RNN_dyn(params=RNN_params).float()
    
model.init_esn_weights(reservoir = True)
model.dt = 0.1
model.eps_int = 1e-4
model.alpha=3
model.max_steps=40
model.save_activations = False

N_dim = model.count_parameters()

loss = nn.CrossEntropyLoss()
# learning parameters

init_pos = model.get_params()

if init_pos.requires_grad:
    # Detach the tensor from the computation graph
    init_pos = init_pos.detach()
if init_pos.is_cuda:
    # Move the tensor to the CPU
    init_pos = init_pos.cpu()
init_pos = init_pos.numpy()

pop_size = int(0.01*N_dim)
PEPG_optimizer = PEPG_opt(N_dim, pop_size = pop_size, learning_rate=0.01, starting_mu=init_pos ,starting_sigma=1e-1)

PEPG_optimizer.sigma_decay = 0.9999
PEPG_optimizer.sigma_alpha=0.2
PEPG_optimizer.sigma_limit=0.01
PEPG_optimizer.elite_ratio=0.1
PEPG_optimizer.weight_decay=0.005
        
#print(f'Using {n_neurons} per layer, run {s+1}, number of parameters {model.count_parameters()}')
#D = train_online_pop_NN(model, n_epochs, train_loader_MNIST, test_loader_MNIST, loss, PEPG_optimizer)
#results.append(D)

In [4]:
def build_batched_params_from_coord(coord, model, all_params, dtype=torch.float32, device=None):
    """
    Convert a population matrix of flattened trainable parameters into 
    a list of batched parameters for functorch vmap.

    Parameters:
    - coord: np.ndarray of shape (population_size, n_trainable_params)
    - model: the torch model (used to query parameter structure)
    - all_params: list of all model parameters (from make_functional_with_buffers)
    - dtype: torch dtype to cast (default: torch.float32)
    - device: device to move tensors to (default: None — keeps current device)

    Returns:
    - batched_params: list of tensors, each of shape (population_size, param_shape...)
    """

    # Step 1: Build a mask of trainable (requires_grad) parameters
    mask_requires_grad = [p.requires_grad for p in model.parameters()]

    # Step 2: Get shapes and sizes of trainable parameters
    trainable_param_shapes = [p.shape for p, keep in zip(all_params, mask_requires_grad) if keep]
    trainable_param_sizes = [np.prod(shape) for shape in trainable_param_shapes]
    cum_sizes = np.cumsum([0] + trainable_param_sizes)

    # Step 3: Unflatten a single flat row to list of tensors (trainable only)
    def unflatten_trainable(flat):
        return [
            torch.tensor(flat[cum_sizes[i]:cum_sizes[i+1]], dtype=dtype, device=device).view(trainable_param_shapes[i])
            for i in range(len(trainable_param_shapes))
        ]

    # Step 4: Rebuild the full param list with fixed params reinserted
    def insert_into_full_params(trainable_param_list):
        full_list = []
        idx = 0
        for keep, p in zip(mask_requires_grad, all_params):
            if keep:
                full_list.append(trainable_param_list[idx])
                idx += 1
            else:
                full_list.append(p)  # Keep frozen param as-is
        return full_list

    # Step 5: Process entire population
    full_param_sets = [
        insert_into_full_params(unflatten_trainable(coord[i]))
        for i in range(coord.shape[0])
    ]

    # Step 6: Stack by parameter index (list of tensors, each of shape [pop_size, ...])
    batched_params = [
        torch.stack([full_param_sets[i][j] for i in range(coord.shape[0])])
        for j in range(len(all_params))
    ]

    return batched_params

In [5]:
# Get candidate parameter matrix (float32)
coordinates = PEPG_optimizer.ask()

In [8]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move model to device
model = model.to(device)

# Make functional and move buffers/params to device
fmodel, all_params, buffers = make_functional_with_buffers(model)
all_params = [p.to(device) for p in all_params]
buffers = [b.to(device) for b in buffers]

# Vectorized model
batched_forward = vmap(fmodel, in_dims=(0, None, None))


# Move input to device
X_train_MNIST = X_train_MNIST.float().to(device)

# Build batched parameters on GPU
start_time = time.time()
# Forward pass on GPU
with torch.no_grad():
    
    batched_params = build_batched_params_from_coord(coordinates, model, all_params,device=device)
    Y_pred_parallel = batched_forward(batched_params, buffers, X_train_MNIST)

end_time = time.time()
print(f'Parallel forward pass time: {end_time - start_time:.4f} seconds')

Using device: cuda


C:\Users\Admin\AppData\Local\Temp\ipykernel_16556\1515456032.py:9: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.make_functional_with_buffers` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.func.functional_call` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  fmodel, all_params, buffers = make_functional_with_buffers(model)
C:\Users\Admin\AppData\Local\Temp\ipykernel_16556\1515456032.py:14: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  bat

Parallel forward pass time: 0.2360 seconds


In [12]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move model to device
model = model.to(device)


# Move input to device
X_train_MNIST = X_train_MNIST.float().to(device)

# Build batched parameters on GPU
Y_pred = torch.zeros([coordinates.shape[0],1000,10],device=device)
start_time = time.time()

# Forward pass on GPU
with torch.no_grad():
    for k in range(coordinates.shape[0]):
                    
        Y_pred[k,:,:] = model.forward_pass_params(coordinates[k,:],X_train_MNIST)

end_time = time.time()
print(f'Regular forward pass time: {end_time - start_time:.4f} seconds')

Using device: cuda
Regular forward pass time: 4.6927 seconds


In [8]:
4.6/0.3

15.333333333333332

In [20]:
diff_outputs = 100*torch.abs(Y_pred_parallel-Y_pred)/Y_pred

In [21]:
torch.mean(diff_outputs)

tensor(6.8666e-06, device='cuda:0')

C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\314129472.py:3: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  batched_forward = vmap(fmodel, in_dims=(0, None, None))
